In [10]:
import pyarrow as pa
import pyarrow.parquet as pq
import json
import os


In [ ]:
input_folder = r"G:\coding\python\web_scraping\Youm7\youm7_scrape\extracted_information"
output_folder = r"G:\coding\python\web_scraping\Youm7\youm7_scrape\data\raw"

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

files = []

# Collect all JSONL files
for file_name in os.listdir(input_folder):
    if file_name.endswith(".jsonl"):
        full_path = os.path.join(input_folder, file_name)
        files.append(full_path)
files

['G:\\coding\\python\\web_scraping\\Youm7\\youm7_scrape\\extracted_information\\arab.jsonl',
 'G:\\coding\\python\\web_scraping\\Youm7\\youm7_scrape\\extracted_information\\art.jsonl',
 'G:\\coding\\python\\web_scraping\\Youm7\\youm7_scrape\\extracted_information\\caricature.jsonl',
 'G:\\coding\\python\\web_scraping\\Youm7\\youm7_scrape\\extracted_information\\economy.jsonl',
 'G:\\coding\\python\\web_scraping\\Youm7\\youm7_scrape\\extracted_information\\investigations.jsonl',
 'G:\\coding\\python\\web_scraping\\Youm7\\youm7_scrape\\extracted_information\\politics.jsonl',
 'G:\\coding\\python\\web_scraping\\Youm7\\youm7_scrape\\extracted_information\\reports.jsonl',
 'G:\\coding\\python\\web_scraping\\Youm7\\youm7_scrape\\extracted_information\\television.jsonl',
 'G:\\coding\\python\\web_scraping\\Youm7\\youm7_scrape\\extracted_information\\urgent.jsonl',
 'G:\\coding\\python\\web_scraping\\Youm7\\youm7_scrape\\extracted_information\\your_horoscope_today.jsonl']

In [12]:
schema = pa.schema([
    ("article_id", pa.string()),
    ("url", pa.string()),
    ("category", pa.string()),
    ("title", pa.string()),
    ("publish_date", pa.string()),
    ("author", pa.string()),
    ("content", pa.string()),
    ("tags", pa.list_(pa.string())),
    ("images", pa.list_(
        pa.struct([
            ("url", pa.string()),
            ("alt", pa.string()),
            ("caption", pa.string()),
            ("type", pa.string())
        ])
    ))
])

In [13]:
def normalize_record(r):
    # tags
    if not r.get("tags"):
        r["tags"] = []
    else:
        r["tags"] = [str(t).strip() for t in r["tags"]]

    # images
    if not r.get("images"):
        r["images"] = []
    else:
        fixed_images = []
        for img in r["images"]:
            fixed_images.append({
                "url": img.get("url", ""),
                "alt": img.get("alt", ""),
                "caption": img.get("caption", ""),
                "type": img.get("type", "")
            })
        r["images"] = fixed_images

    return r

In [14]:
def jsonl_to_parquet_stream(input_file, output_file, chunk_size=5000):
    writer = None
    batch = []

    with open(input_file, "r", encoding="utf-8") as f:
        for line in f:
            r = normalize_record(json.loads(line))
            batch.append(r)

            if len(batch) == chunk_size:
                table = pa.Table.from_pylist(batch, schema=schema)

                if writer is None:
                    writer = pq.ParquetWriter(output_file, schema)

                writer.write_table(table)
                batch = []

        if batch:
            table = pa.Table.from_pylist(batch, schema=schema)
            if writer is None:
                writer = pq.ParquetWriter(output_file, schema)
            writer.write_table(table)

    if writer:
        writer.close()

In [15]:

# Process each file
for file_path in files:
    file_name = os.path.basename(file_path).replace(".jsonl", "")
    output_path = os.path.join(output_folder, f"{file_name}.parquet")

    print(f"Processing: {file_name}")
    jsonl_to_parquet_stream(file_path, output_path)
    print(f"Done: {output_path}\n")

Processing: arab
Done: G:\coding\python\web_scraping\Youm7\data\raw\arab.parquet

Processing: art
Done: G:\coding\python\web_scraping\Youm7\data\raw\art.parquet

Processing: caricature
Done: G:\coding\python\web_scraping\Youm7\data\raw\caricature.parquet

Processing: economy
Done: G:\coding\python\web_scraping\Youm7\data\raw\economy.parquet

Processing: investigations
Done: G:\coding\python\web_scraping\Youm7\data\raw\investigations.parquet

Processing: politics
Done: G:\coding\python\web_scraping\Youm7\data\raw\politics.parquet

Processing: reports
Done: G:\coding\python\web_scraping\Youm7\data\raw\reports.parquet

Processing: television
Done: G:\coding\python\web_scraping\Youm7\data\raw\television.parquet

Processing: urgent
Done: G:\coding\python\web_scraping\Youm7\data\raw\urgent.parquet

Processing: your_horoscope_today
Done: G:\coding\python\web_scraping\Youm7\data\raw\your_horoscope_today.parquet

